# Transformation explorer

Plots a TIMIT WAV file's raw waveform next to its spectrogram, featurized like `src/debug_items.py` does
(`compute_mel_spectrogram` / `compute_linear_spectrogram` from `src/data/features.py`, then `apply_drc`).
The widgets start at the `TransformationConfig` defaults; changing any of them (or the WAV file) re-does the transformation.

In [4]:
import html
import sys
from dataclasses import replace
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import soundfile as sf
from IPython.display import Audio, display
from plotly.subplots import make_subplots
import plotly.io as pio; pio.renderers.default = "notebook"

ROOT = Path.cwd().parent  # notebook runs from doc/
sys.path.insert(0, str(ROOT))

from src.config import TransformationConfig
from src.data.features import apply_drc, compute_linear_spectrogram, compute_mel_spectrogram

In [5]:
timit_root = Path("~/.cache/datasets/timit").expanduser()
# Labelled relative to the TIMIT/ folder, e.g. "TRAIN/DR1/FCJF0/SA1.WAV".
wav_paths = [(str(p.relative_to(p.parents[3])), p) for p in sorted(timit_root.rglob("*")) if p.suffix.lower() == ".wav"]
len(wav_paths)

6300

In [6]:
defaults = TransformationConfig()
wav_path = widgets.Dropdown(options=wav_paths, description="WAV file", layout=widgets.Layout(width="500px"))
frame_length_s = widgets.FloatText(value=defaults.frame_length_s, step=0.001, description="frame_length_s")
frame_step_s = widgets.FloatText(value=defaults.frame_step_s, step=0.001, description="frame_step_s")
nfft = widgets.IntText(value=defaults.nfft, description="nfft")
n_mels = widgets.IntText(value=defaults.n_mels, description="n_mels")
fmin = widgets.BoundedFloatText(value=defaults.fmin, min=0, max=defaults.sample_rate, description="fmin")
fmax = widgets.BoundedFloatText(value=defaults.fmax, min=defaults.fmin, max=defaults.sample_rate, description="fmax")
type_ = widgets.Dropdown(options=[("mel", "mel"), ("lin", "linear")], value=defaults.type, description="type")
window = widgets.Dropdown(options=["hann", "hamming"], value=defaults.window, description="window")

# Keep fmin <= fmax: each bounds the other.
widgets.dlink((fmin, "value"), (fmax, "min"))
widgets.dlink((fmax, "value"), (fmin, "max"))


# The figure (as an iframe) and the audio player go into HTML widgets: plotly's and Audio's own
# notebook outputs don't render inside an Output widget in PyCharm.
plot = widgets.HTML()
player = widgets.HTML()


def show(wav_path, frame_length_s, frame_step_s, nfft, n_mels, fmin, fmax, type_, window):
    waveform, sample_rate = sf.read(str(wav_path), dtype="float32")
    assert sample_rate == defaults.sample_rate, f"{wav_path} is {sample_rate} Hz, config expects {defaults.sample_rate} Hz"
    config = replace(defaults, frame_length_s=frame_length_s, frame_step_s=frame_step_s, nfft=nfft,
                     n_mels=n_mels, fmin=fmin, fmax=fmax, type=type_, window=window)
    # FFT magnitude of every frame (the linear spectrogram); the mel spectrogram is computed from it.
    stft = apply_drc(compute_linear_spectrogram(waveform, config))
    if type_ == "linear":
        features = stft
    else:
        features = apply_drc(compute_mel_spectrogram(waveform, config))

    # Plotly, so hovering shows the exact values at the mouse position.
    title = f"waveform: {wav_path.relative_to(wav_path.parents[3])}"
    freq_label = "mel bin" if type_ == "mel" else "freq bin"
    frame_times = np.arange(features.shape[0]) * config.frame_step_s
    fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.07, subplot_titles=[
        title, f"FFT per frame: {stft.shape[0]} frames x {stft.shape[1]} bins",
        f"{type_} spectrogram {features.shape[0]} frames x {features.shape[1]} bins"])
    fig.add_trace(go.Scattergl(
        x=np.arange(len(waveform)) / sample_rate, y=waveform, mode="lines", line_width=0.5, showlegend=False,
        hovertemplate="time: %{x:.4f} s<br>amplitude: %{y:.4f}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=stft.T, x=frame_times, y=np.arange(stft.shape[1]) * sample_rate / config.nfft,
        colorscale="Magma", colorbar=dict(title_text="DRC level", x=fig.layout.xaxis2.domain[1] + 0.005),
        hovertemplate="time: %{x:.3f} s<br>frequency: %{y:.1f} Hz<br>DRC level: %{z:.2f}<extra></extra>",
    ), row=1, col=2)
    fig.add_trace(go.Heatmap(
        z=features.T, x=frame_times, y=np.arange(features.shape[1]),
        colorscale="Magma", colorbar_title_text="DRC level",
        hovertemplate=f"time: %{{x:.3f}} s<br>{freq_label}: %{{y}}<br>DRC level: %{{z:.2f}}<extra></extra>",
    ), row=1, col=3)
    if type_ == "mel":
        # The part of the FFT spectrum the mel filterbank spreads its filters over.
        for name, f in [("fmin", fmin), ("fmax", fmax)]:
            fig.add_hline(y=f, line=dict(color="cyan", width=1, dash="dash"), annotation_text=name,
                          annotation_font_color="cyan", row=1, col=2)
    fig.update_xaxes(title_text="time [s]")
    fig.update_yaxes(title_text="amplitude", row=1, col=1)
    fig.update_yaxes(title_text="frequency [Hz]", row=1, col=2)
    fig.update_yaxes(title_text=freq_label, row=1, col=3)
    fig.update_layout(height=400, margin=dict(l=60, r=20, t=40, b=50))
    page = fig.to_html(include_plotlyjs="cdn", config=dict(responsive=True))
    plot.value = f'<iframe srcdoc="{html.escape(page)}" style="width: 100%; height: 420px; border: 0"></iframe>'
    player.value = Audio(waveform, rate=sample_rate)._repr_html_()


output = widgets.interactive_output(show, dict(
    wav_path=wav_path, frame_length_s=frame_length_s, frame_step_s=frame_step_s, nfft=nfft, n_mels=n_mels,
    fmin=fmin, fmax=fmax, type_=type_, window=window,
))
display(widgets.VBox([
    wav_path,
    widgets.HBox([frame_length_s, frame_step_s, nfft, n_mels]),
    widgets.HBox([fmin, fmax, type_, window]),
    plot,
    player,
    output,
]))
